In [1]:
# ----- 11 vertex pattern -> 6 vertex + 5 hyperedge -----------------------------------------------

import itertools
import time
start_time = time.time()

# ----- Step 1: Pattern details --------------------------------------------------------------------

vertex_in_pattern = 6
edge_in_pattern = 5
maximum_hyperedge_size = 3
main_filter_degree = 2

# ----- Step 2: Generate all possible hyperedges ---------------------------------------------------

vertices = list(range(vertex_in_pattern))
all_possible_edges = []
for r in range(2, maximum_hyperedge_size+1): 
    all_possible_edges.extend(itertools.combinations(vertices, r))

# ----- Step 3: Validation functions ---------------------------------------------------------------

def validate_degree(edges):
    # Initialize degree count to zero for all vertices
    degree = {v: 0 for v in vertices}
    
    # Count degrees for all vertices    
    for e in edges:
        for v in e:
            if v in degree:
                degree[v] += 1

    # Check if all vertices have degree at least 2
    for v in vertices:
        if degree[v] < 2:
            return False
    return True

def validate_edge_subset(edges):
    # Convert edges to sets once
    edge_sets = [set(e) for e in edges]

    # Compare each pair of edges
    n = len(edge_sets)
    for i in range(n):
        e1 = edge_sets[i]
        for j in range(n):
            if i != j:
                e2 = edge_sets[j]
                if e1 <= e2:  # same as e1.issubset(e2)
                    return False

    return True  # no edge is a subset of another

def validate_vertex_subset(edges):
    incident_edges = {
        v: {frozenset(e) for e in edges if v in e}
        for v in vertices
    }

    for i in range(len(vertices)):
        u = vertices[i]
        eu = incident_edges[u]

        for j in range(i + 1, len(vertices)):
            v = vertices[j]
            ev = incident_edges[v]

            if len(eu) <= len(ev) and eu.issubset(ev):
                return False
            if len(ev) <= len(eu) and ev.issubset(eu):
                return False

    return True
    
# ----- Step 4: Filter functions -------------------------------------------------------------------------------

def validate_no_full_union(edges):
    all_verts = set(vertices)
    edge_sets = [set(e) for e in edges]

    for i, e1 in enumerate(edge_sets):
        for j, e2 in enumerate(edge_sets):
            if i < j and e1.union(e2) == all_verts:
                return False
    return True
    
def main_filter(edges):
    all_verts = set(vertices)
    edge_sets = [set(e) for e in edges]

    for v in vertices:
        incident_count = sum(1 for e in edge_sets if v in e)
        if incident_count <= main_filter_degree:
            remaining_verts = all_verts - {v}
            edges_without_v = [e for e in edge_sets if v not in e]
    
            for i in range(len(edges_without_v)):
                e1 = edges_without_v[i]
                missing_after_e1 = remaining_verts - e1
    
                for j in range(i + 1, len(edges_without_v)):
                    e2 = edges_without_v[j]
                    if missing_after_e1.issubset(e2):
                        return False
    return True

# ----- Step 5: Canonical representation ----------------------------------------------------------------------------

def canonical_form_edges(H):
    """
    Returns a canonical representation of a hypergraph
    invariant under vertex relabeling.
    """

    # normalize edges
    H = [tuple(sorted(e)) for e in H]

    # extract used vertices
    vertices = sorted({v for e in H for v in e})

    best = None

    for perm in itertools.permutations(vertices):
        mapping = dict(zip(vertices, perm))

        relabeled = [
            tuple(sorted(mapping[v] for v in e))
            for e in H
        ]
        relabeled.sort()
        rep = tuple(relabeled)

        if best is None or rep < best:
            best = rep

    return best

def is_new_hypergraph(edge_choice, canonical_seen):
    canon = canonical_form_edges(edge_choice)
    return canon not in canonical_seen, canon


# ----- Step 6: Generate all hypergraphs ------------------------------------------------------------------------------

total_hgs = itertools.combinations(all_possible_edges, edge_in_pattern)
print("Generating hypergraphs lazily...")

unique_graphs_seen = []     # stores original edge lists (h1, h2, h3, ...)
canonical_seen = set()      # stores canonical fingerprints
valid_count = 0

def validate_pattern(H):
    return validate_degree(H) and validate_edge_subset(H) and validate_vertex_subset(H) and validate_no_full_union(H) and main_filter(H)
  
for idx, edge_choice in enumerate(total_hgs, start=1):

    if validate_pattern(edge_choice):
        is_new, canon = is_new_hypergraph(edge_choice, canonical_seen)
    
        if is_new:
            canonical_seen.add(canon)
            unique_graphs_seen.append(edge_choice)
            print(f"[VALID]   #{idx}: {edge_choice}")
            valid_count += 1

print(f"\nTotal unique valid hypergraphs: {valid_count}")
end_time = time.time()
elapsed = end_time - start_time
print(f"Time taken: {elapsed:.4f} seconds")


Generating hypergraphs lazily...

Total unique valid hypergraphs: 0
Time taken: 1.4538 seconds
